In [1]:
def generate_summary_table(raw_df):
    # Group by gear position and calculate min/max for vehicle and engine speeds
    summary = raw_df.groupby('transmission_gear_position').agg({
        'vehicle_speed': ['min', 'max'],
        'engine_speed': ['min', 'max']
    }).reset_index()

    # Flatten MultiIndex columns
    summary.columns = ['Gear', 'Min Vehicle Speed', 'Max Vehicle Speed', 'Min Engine Speed', 'Max Engine Speed']

    return summary

def check_data_compliance(raw_df, syn_df):
    max_speed = raw_df['vehicle_speed'].max()
    
    
    def calculate_violations(df):
        total_rows = df.shape[0]

        # Rule 1: Vehicle speed within a reasonable range
        violations_speed = df[(df['vehicle_speed'] < 0) | (df['vehicle_speed'] > max_speed)].shape[0]

        # Rule 2: Reasonable gear changes
        def gear_to_num(gear):
            # Define the mapping from gear position to a numerical value
            mapping = {
                'first': 1,
                'second': 2,
                'third': 3,
                'fourth': 4,
                'fifth': 5,
                'sixth': 6,
                'neutral': 0,
                'reverse': -1,
                'nuetral': 0
            }
            return mapping.get(gear, 0)  # Default to 0 for unknown gears

        df['gear_num'] = df['transmission_gear_position'].apply(gear_to_num)
        df['gear_change'] = df['gear_num'].diff().abs()
        violations_gear = df[df['gear_change'] > 2].shape[0]

        # Rule 3: Matching engine speed, vehicle speed, and gear position
        # Placeholder for now, as specific criteria are needed
        def vs_es_gear_match(syn_df):
            summary_table = generate_summary_table(raw_df)
            lookup_dict = summary_table.set_index('Gear').to_dict('index')

            conditions = [
                (syn_df['transmission_gear_position'] == gear) &
                (syn_df['vehicle_speed'].between(lookup_dict[gear]['Min Vehicle Speed'], lookup_dict[gear]['Max Vehicle Speed'])) &
                (syn_df['engine_speed'].between(lookup_dict[gear]['Min Engine Speed'], lookup_dict[gear]['Max Engine Speed']))
                for gear in lookup_dict
            ]
        
            # Combine conditions
            overall_condition = pd.concat([syn_df[condition] for condition in conditions], axis=0).drop_duplicates()
        
            # Calculate the proportion of violations
            violations_matching = syn_df.shape[0] - overall_condition.shape[0]
        
            return violations_matching

        violations_matching = vs_es_gear_match(df)
        

        # Rule 4: Vehicle speed and engine speed increase/decrease simultaneously
        df['vehicle_speed_change'] = df['vehicle_speed'].diff()
        df['engine_speed_change'] = df['engine_speed'].diff()
        violations_speed_direction = df[(df['vehicle_speed_change'] * df['engine_speed_change'] < 0) & (df['vehicle_speed_change'] != 0) & (df['engine_speed_change'] != 0)].shape[0]

        return {
            'speed_range_violations': violations_speed / total_rows,
            'gear_change_violations': violations_gear / total_rows,
            'matching_violations': violations_matching / total_rows,
            'speed_direction_violations': violations_speed_direction / total_rows
        }

    # Calculate violations for both datasets
    raw_violations = calculate_violations(raw_df)
    syn_violations = calculate_violations(syn_df)

    # return raw_violations, syn_violations

    return [1-v for k, v in syn_violations.items()]

In [2]:
import pandas as pd

raw_df = pd.read_csv("../data_selected/openxc/nyc_downtown_east.csv")
syn_df = pd.read_csv("../results/vehiclesec2024/small-scale/csv/realtabformer-tabular_openxc-nyc-downtown-east_20231130183157205073649.csv")

In [3]:
print(check_data_compliance(raw_df, syn_df))

[0.9999968685707844, 1.0, 0.9492771095655769, 0.7721415531262623]


In [4]:
def summarize_gear_speed_ranges(csv_path):
    # Load the CSV file
    df = pd.read_csv(csv_path)

    # Group by gear position and calculate min/max for vehicle and engine speeds
    summary = df.groupby('transmission_gear_position').agg({
        'vehicle_speed': ['min', 'max'],
        'engine_speed': ['min', 'max']
    }).reset_index()

    # Rename columns for clarity
    summary.columns = ['Gear', 'Min Vehicle Speed', 'Max Vehicle Speed', 'Min Engine Speed', 'Max Engine Speed']

    return summary

summarize_gear_speed_ranges("../data_selected/openxc/nyc_downtown_east.csv")

,Gear,Min Vehicle Speed,Max Vehicle Speed,Min Engine Speed,Max Engine Speed
0,first,0.000000,23.490000,0.0,3076.0
1,fourth,24.539999,49.340000,748.0,3068.0
2,neutral,0.000000,0.000000,766.0,780.0
3,second,11.730000,39.000000,954.0,3406.0
4,third,11.070000,42.969997,732.0,3148.0


In [5]:
import os

CANGen_BASE_FOLDER = '/storage/CANGen'

DICT_DATASET_FILENAME = {
    # OpenXC
    'openxc-nyc-downtown-east': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'nyc_downtown_east.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'india_New_Delhi_Railway_to_AIIMS.csv'
    ),
    'openxc-taiwan-highwayno2-can': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'taiwan_HighwayNo2_can.csv'
    ),

    # OpenXC (Sessionized)
    'openxc-nyc-downtown-east-sessionized': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'nyc_downtown_east_sessionized.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims-sessionized': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'india_New_Delhi_Railway_to_AIIMS_sessionized.csv'
    ),
    'openxc-taiwan-highwayno2-can-sessionized': os.path.join(
        CANGen_BASE_FOLDER, 'data_selected', 'openxc', 'taiwan_HighwayNo2_can_sessionized.csv'
    ),

    'openxc-nyc-downtown-east-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'nyc', 'downtown-east', 'downtown-east_before_imputation.csv'
    ),
    'openxc-india-new-delhi-railway-to-aiims-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'india', 'New_Delhi_Railway_to_AIIMS', 'New_Delhi_Railway_to_AIIMS_before_imputation.csv'
    ),
    'openxc-taiwan-highwayno2-can-no-imputation': os.path.join(
        CANGen_BASE_FOLDER, 'data', 'openxc', 'taiwan', 'HighwayNo2-can', 'HighwayNo2-can_before_imputation.csv'
    )
}

In [21]:
from collections import OrderedDict

import numpy as np

query_results = OrderedDict()

for (model, dataset_option) in [
    ('realtabformer-tabular', '-no-imputation'),
    ('realtabformer-tabular', ''),
    ('ctgan', ''),
    ('realtabformer-timeseries', '-sessionized'),
    ('netshare', '-sessionized'),
    ('tabddpm', '')
]:
    if model == "realtabformer-tabular" and dataset_option == "-no-imputation":
        query_results["realtabformer-tabular-raw"] = {}
        model_name = "realtabformer-tabular-raw"
    else:
        query_results[model] = {}
        model_name = model
        
    for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
        raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])

        csv_files = []
        res = []
        for csv_file in os.listdir("../results/vehiclesec2024/small-scale/csv_selected"):
            if csv_file.startswith(f"{model}_{dataset}{dataset_option}_"):
                csv_files.append(csv_file)
                syn_df = pd.read_csv(f"../results/vehiclesec2024/small-scale/csv_selected/{csv_file}").fillna(method='ffill').dropna()
                res.append(check_data_compliance(raw_df, syn_df))
        assert len(csv_files) == 3
        
        print(model, dataset_option, csv_files, "\n")

        num_queries = len(res[0])
        num_runs = len(res)

        res_pm = []
        for i in range(num_queries):
            vals = []
            for j in range(num_runs):
                vals.append(res[j][i])
            res_pm.append((np.mean(vals),np.std(vals)))
        assert len(res_pm) == num_queries

        query_results[model_name][dataset] = res_pm


        

realtabformer-tabular -no-imputation ['realtabformer-tabular_openxc-nyc-downtown-east-no-imputation_20231130183122266036071.csv', 'realtabformer-tabular_openxc-nyc-downtown-east-no-imputation_20231130183140961707714.csv', 'realtabformer-tabular_openxc-nyc-downtown-east-no-imputation_20231201175512946329020.csv'] 

realtabformer-tabular -no-imputation ['realtabformer-tabular_openxc-india-new-delhi-railway-to-aiims-no-imputation_20231130183150778317466.csv', 'realtabformer-tabular_openxc-india-new-delhi-railway-to-aiims-no-imputation_20231130183124276658707.csv', 'realtabformer-tabular_openxc-india-new-delhi-railway-to-aiims-no-imputation_20231130183143476951834.csv'] 

realtabformer-tabular -no-imputation ['realtabformer-tabular_openxc-taiwan-highwayno2-can-no-imputation_20231130183126757084016.csv', 'realtabformer-tabular_openxc-taiwan-highwayno2-can-no-imputation_20231130183151801068265.csv', 'realtabformer-tabular_openxc-taiwan-highwayno2-can-no-imputation_20231130183144495828601.csv

# Export to LaTex table

In [22]:
for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
    for query_index in range(4):
        res = []
        for model_name in ["realtabformer-tabular-raw", "realtabformer-tabular", 'ctgan', 'realtabformer-timeseries', 'netshare', 'tabddpm']:
            res.append(f'${query_results[model_name][dataset][query_index][0]:.2f} \pm {query_results[model_name][dataset][query_index][1]:.2f}$')

        print(dataset, query_index)
        print(" & ".join(res))
        print()

openxc-nyc-downtown-east 0
$1.00 \pm 0.00$ & $1.00 \pm 0.00$ & $0.83 \pm 0.03$ & $1.00 \pm 0.00$ & $1.00 \pm 0.00$ & $1.00 \pm 0.00$

openxc-nyc-downtown-east 1
$1.00 \pm 0.00$ & $1.00 \pm 0.00$ & $0.85 \pm 0.01$ & $0.99 \pm 0.00$ & $0.96 \pm 0.01$ & $0.76 \pm 0.00$

openxc-nyc-downtown-east 2
$0.49 \pm 0.01$ & $0.95 \pm 0.00$ & $0.63 \pm 0.01$ & $0.99 \pm 0.00$ & $0.98 \pm 0.01$ & $0.01 \pm 0.00$

openxc-nyc-downtown-east 3
$0.99 \pm 0.00$ & $0.77 \pm 0.00$ & $0.71 \pm 0.01$ & $0.83 \pm 0.05$ & $0.67 \pm 0.01$ & $0.99 \pm 0.00$

openxc-india-new-delhi-railway-to-aiims 0
$1.00 \pm 0.00$ & $1.00 \pm 0.00$ & $0.90 \pm 0.03$ & $1.00 \pm 0.00$ & $1.00 \pm 0.00$ & $1.00 \pm 0.00$

openxc-india-new-delhi-railway-to-aiims 1
$1.00 \pm 0.00$ & $0.98 \pm 0.00$ & $0.73 \pm 0.01$ & $0.92 \pm 0.01$ & $0.88 \pm 0.04$ & $0.60 \pm 0.00$

openxc-india-new-delhi-railway-to-aiims 2
$0.00 \pm 0.00$ & $1.00 \pm 0.00$ & $0.84 \pm 0.03$ & $0.98 \pm 0.00$ & $0.97 \pm 0.01$ & $0.00 \pm 0.00$

openxc-india-new-

In [62]:
for model in ['realtabformer-tabular', 'ctgan']:
    for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
        raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])
        res = []
        for csv_file in os.listdir("../results/vehiclesec2024/small-scale/csv"):
            if not csv_file.endswith(".csv"):
                continue
            if model == 'realtabformer-tabular' and (not csv_file.startswith(f"{model}_{dataset}_20231130")):
                continue

            if csv_file.startswith(f"{model}_{dataset}_2023"):
                # print(dataset, model, csv_file)

            
                syn_df = pd.read_csv(os.path.join("../results/vehiclesec2024/small-scale/csv", csv_file))

                print(model, dataset)
                print(check_data_compliance(raw_df, syn_df))

realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.00', 'matching_violations': ' 0.05', 'speed_direction_violations': ' 0.23'}
realtabformer-tabular openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.05', 'matching_violations': ' 0.00', 'speed_direction_violations': ' 0.22'}
realtabformer-tabular openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.05', 'matching_violations': ' 0.00', 'speed_direction_violations': ' 0.22'}
realtabformer-tabular ope

In [7]:
import pandas as pd

for model in ['realtabformer-timeseries', 'netshare', 'tabddpm']:
    # print(model)
    for dataset in ['openxc-nyc-downtown-east', 'openxc-india-new-delhi-railway-to-aiims', 'openxc-taiwan-highwayno2-can']:
        # print(dataset)
        raw_df = pd.read_csv(DICT_DATASET_FILENAME[dataset])
        csv_files = []
        for csv_file in os.listdir("../results/vehiclesec2024/small-scale/csv"):
            if csv_file.startswith(f"{model}_{dataset}"):
                csv_files.append(csv_file)
        csv_files = sorted(csv_files)[-3:]

        syn_df = pd.read_csv(os.path.join("../results/vehiclesec2024/small-scale/csv", csv_files[1]))

        print(model, dataset)
        print(check_data_compliance(raw_df, syn_df))
        print()

realtabformer-timeseries openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.03', 'matching_violations': ' 0.01', 'speed_direction_violations': ' 0.10'}

realtabformer-timeseries openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.14', 'matching_violations': ' 0.02', 'speed_direction_violations': ' 0.16'}

realtabformer-timeseries openxc-taiwan-highwayno2-can
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.03', 'matching_violations': ' 0.02', 'speed_direction_violations': ' 0.18'}

netshare openxc-nyc-downtown-east
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.12', 'matching_violations': ' 0.02', 'speed_direction_violations': ' 0.33'}

netshare openxc-india-new-delhi-railway-to-aiims
{'speed_range_violations': ' 0.00', 'gear_change_violations': ' 0.17', 'matching_violations': ' 0.04', 'speed_direction_violations': ' 0.34'}

netshare openxc-taiwan-highwayno2